# Improving the four models on dataset `01_original`

The baseline is the team notebook `04_denoise_clahe_dino.ipynb` run unchanged on 01 (results in `results/01_original/`, read only here).
This notebook tries the improvements from the report's plan, **on dataset 01 only**:

| section | model | what changes |
|---|---|---|
| A | Classic SVM | feature ablation, balanced feature groups, new colour/texture features |
| B | CLIP | ViT-L/14, class-description prompts, flip TTA, linear probe (new "few-shot" row) |
| C | Ensemble | CLIP + ResNet-50, weighted log-probabilities |
| D | CNN | 100 epochs + stronger augmentation; residual blocks |
| E | ResNet-50 | layer-wise LR + warmup + stronger augmentation; 288 px |

Rules: every choice uses **val** only; test is scored once per config and reported with a paired test against the baseline.
Cells 0–3 below are copied verbatim from the team notebook, so data checks, metrics and saving are identical.
Results go to `results/01_original_improved/`; with `RETRAIN = False` a re-run reloads every finished config.

In [ ]:
# ==== the only cell you normally edit ====
DATASET = "01_original"   # "01_original" | "02_denoise_clahe" | "03_denoise_clahe_gsam" | "04_denoise_clahe_dino"
RUN_MODE = "full"                  # "smoke": 20 imgs/class/split, 2 epochs, to test the pipeline | "full": the real run
SEED = 42
RETRAIN = False                     # True: ignore saved checkpoints / grid results and train again

# ---- from here on: identical for every dataset (same as the 03 run) ----
IMG_SIZE = 224
PAD_FILL = (124, 116, 104)          # same grey as the GSAM background (used to pad every dataset to a square)
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LABEL_SMOOTHING = 0.1

CLASSICAL_FEATURES = "handcrafted"  # "handcrafted" (HSV hist + HOG + LBP) | "resnet50" (frozen ImageNet embeddings)
LOGREG_C = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
SVM_C = [0.1, 1, 10, 100, 1000]
SVM_GAMMA_MULT = [0.3, 1, 3]        # x 1/n_features (= sklearn "scale" after standardising)

CNN_GRID = [dict(n_blocks=n, base=32, dropout=0.3) for n in (3, 4, 5)]
CNN_TRAIN = dict(max_epochs=40, patience=6, lr=1e-3, weight_decay=5e-4)

RESNET_GRID = [dict(unfreeze="fc", lr=1e-3), dict(unfreeze="layer4", lr=3e-4), dict(unfreeze="all", lr=1e-4)]
RESNET_TRAIN = dict(max_epochs=15, patience=4, weight_decay=1e-4)

CLIP_ID = "openai/clip-vit-base-patch32"
TEMPLATE_SETS = {
    "classname": ["{}"],
    "a photo of": ["a photo of {}."],
    "food photo": ["a photo of {}, a type of food."],
    "ensemble": ["a photo of {}, a type of food.", "a close-up photo of {}.", "a photo of a plate of {}.",
                 "a good photo of {}.", "a cropped photo of {}."],
}

EXPECTED_MANIFEST_SHA256 = "79eb1d1cf8a475b8deaa2a301acc64a1c3a334fd4a289c8d45af91c359c3c2a4"
SMOKE_PER_CLASS = 20
if RUN_MODE == "smoke":
    CNN_TRAIN = {**CNN_TRAIN, "max_epochs": 2}
    RESNET_TRAIN = {**RESNET_TRAIN, "max_epochs": 2}
    LOGREG_C, SVM_C, SVM_GAMMA_MULT = [1e-2, 1], [1, 10], [1]

In [ ]:
# ==== paths (local VS Code or Colab) ====
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/food_classify")   # upload <DATASET>.zip here
    DATA_ROOT = Path("/content/data/dataset")
    if not (DATA_ROOT / DATASET).exists():
        import zipfile
        with zipfile.ZipFile(DRIVE_DIR / f"{DATASET}.zip") as z:
            z.extractall(DATA_ROOT)
    OUT_ROOT = DRIVE_DIR / "results"                               # on Drive, so it survives a disconnect
else:
    here = Path.cwd().resolve()
    PROJECT = next((p for p in [here, *here.parents] if (p / "data" / "dataset" / DATASET).exists()), None)
    assert PROJECT is not None, f"data/dataset/{DATASET} not found above {here}: unzip the dataset into data/dataset/"
    DATA_ROOT = PROJECT / "data" / "dataset"
    OUT_ROOT = PROJECT / "results"
    os.environ.setdefault("TORCH_HOME", str(PROJECT / "checkpoints" / "torch"))  # reuse downloaded weights
    os.environ.setdefault("HF_HOME", str(PROJECT / "checkpoints" / "hf"))

META = DATA_ROOT / "meta"
OUT = (OUT_ROOT / DATASET) if RUN_MODE == "full" else (OUT_ROOT / "smoke" / DATASET)
CKPT_DIR, FIG_DIR = OUT / "checkpoints", OUT / "figures"
for d in (OUT, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("data   :", DATA_ROOT / DATASET)
print("results:", OUT)

In [ ]:
# outputs of this notebook go next to the baseline, never into it
BASE_OUT = OUT                                   # baseline results of the team notebook (read only)
OUT = OUT.parent / f"{DATASET}_improved"
CKPT_DIR, FIG_DIR = OUT / "checkpoints", OUT / "figures"
for d in (OUT, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert (BASE_OUT / "summary_test.csv").exists(), f"run the baseline first: {BASE_OUT}"
print("baseline:", BASE_OUT)
print("improved:", OUT)

# improvement settings
SVM2_C, SVM2_GAMMA_MULT = [1, 10, 100, 1000], [0.3, 1, 3]
ABLATION_C = [1, 10, 100]
CLIP_IDS = {"B32": CLIP_ID, "L14": "openai/clip-vit-large-patch14"}
PROBE_C = [1e-3, 1e-2, 1e-1, 1, 10]
CNN2_TRAIN = dict(max_epochs=100, patience=10, lr=1e-3, weight_decay=5e-4)
RN2_TRAIN = dict(max_epochs=20, patience=5, weight_decay=1e-4)
if RUN_MODE == "smoke":
    SVM2_C, SVM2_GAMMA_MULT, ABLATION_C, PROBE_C = [10], [1], [10], [1]
    CNN2_TRAIN = {**CNN2_TRAIN, "max_epochs": 2}
    RN2_TRAIN = {**RN2_TRAIN, "max_epochs": 2}

In [ ]:
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import label_binarize

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"                    # fp16 helps on CUDA; it did not help on the M1
NON_BLOCKING = DEVICE.type == "cuda"
NUM_WORKERS = 0 if sys.platform == "darwin" else 2  # macOS Jupyter can't pickle classes defined in a notebook into workers
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")


def finish(fig, title, name):
    """Lay out a grid figure with its title above the first row (no overlap), save, show."""
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / fig.get_figheight()))
    fig.suptitle(title, y=1 - 0.12 / fig.get_figheight())
    savefig(fig, name)
    plt.show()


seed_everything()
print(f"python {sys.version.split()[0]} ({sys.executable}) | torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")

In [ ]:
sha = hashlib.sha256((META / "manifest.csv").read_bytes()).hexdigest()
assert sha == EXPECTED_MANIFEST_SHA256, f"manifest differs from the team's ({sha}): wrong zip?"

df_all = pd.read_csv(META / "manifest.csv")
df_all["path"] = [str(DATA_ROOT / DATASET / r) for r in df_all.relpath]
missing = [p for p in df_all.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} images in the manifest are missing, e.g. {missing[:3]}"
on_disk = {str(p.relative_to(DATA_ROOT / DATASET)) for p in (DATA_ROOT / DATASET).glob("*/*/*.jpg")}
assert on_disk == set(df_all.relpath), "folder contents and manifest differ"
assert (df_all.relpath.str.split("/").str[0] == df_all.split).all() and \
       (df_all.relpath.str.split("/").str[1] == df_all.label).all(), "folder and manifest split/label disagree"

CLASSES = sorted(df_all.label.unique())
K = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
df_all["y"] = df_all.label.map(CLS2IDX)

key = df_all.label + "/" + df_all.relpath.str.split("/").str[-1]
assert not key.duplicated().any(), "the same image id appears in more than one split"

counts = df_all.pivot_table(index="label", columns="split", values="relpath", aggfunc="count")[["train", "val", "test"]]
counts.loc["total"] = counts.sum()
display(counts)
assert counts.drop("total").nunique().eq(1).all(), "classes are not balanced"
print("split ratio:", (counts.loc["total"] / counts.loc["total"].sum()).round(3).to_dict())

df = df_all.groupby(["split", "label"], group_keys=False).head(SMOKE_PER_CLASS) if RUN_MODE == "smoke" else df_all
splits = {s: df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y_split = {s: v.y.values for s, v in splits.items()}
print({s: len(v) for s, v in splits.items()}, "images used in this run")

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLIP_MEAN, CLIP_STD = [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]


def pad_to_square(img, fill=PAD_FILL):
    w, h = img.size
    s = max(w, h)
    out = Image.new("RGB", (s, s), fill)
    out.paste(img, ((s - w) // 2, (s - h) // 2))
    return out


train_tf = T.Compose([T.Lambda(pad_to_square),
                      T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
                      T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = T.Compose([T.Lambda(pad_to_square), T.Resize((IMG_SIZE, IMG_SIZE)),
                     T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
clip_tf = T.Compose([T.Lambda(pad_to_square), T.Resize((IMG_SIZE, IMG_SIZE)),
                     T.ToTensor(), T.Normalize(CLIP_MEAN, CLIP_STD)])


def show_tensor(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Normalised CHW tensor -> HWC image in [0, 1] for plotting."""
    x = t.cpu() * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


class FoodDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame.path.tolist(), frame.y.tolist(), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


def make_loader(split, transform, shuffle=False, batch_size=None):
    return DataLoader(FoodDataset(splits[split], transform), batch_size=batch_size or (BATCH_SIZE if shuffle else EVAL_BATCH_SIZE),
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)


# what the models see
demo = splits["train"].sample(3, random_state=SEED)
fig, axes = plt.subplots(3, 5, figsize=(12, 7.5))
for row, (_, r) in zip(axes, demo.iterrows()):
    img = Image.open(r.path).convert("RGB")
    row[0].imshow(img); row[0].set_title(f"stored {img.size}", fontsize=8)
    row[1].imshow(show_tensor(eval_tf(img))); row[1].set_title("eval 224x224", fontsize=8)
    for ax in row[2:]:
        ax.imshow(show_tensor(train_tf(img))); ax.set_title("train aug", fontsize=8)
    for ax in row: ax.axis("off")
plt.tight_layout(); savefig(fig, "transforms"); plt.show()

In [ ]:
RESULTS = {}   # model name -> {"val": {...}, "test": {...}, "config": {...}}


def topk_accuracy(y, scores, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return float((top == y[:, None]).any(axis=1).mean())


def compute_metrics(y, scores):
    y = np.asarray(y)
    pred = scores.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    Y = label_binarize(y, classes=np.arange(K))
    ap = np.array([average_precision_score(Y[:, c], scores[:, c]) for c in range(K)])
    pc_p, pc_r, pc_f, support = precision_recall_fscore_support(y, pred, labels=np.arange(K), zero_division=0)
    per_class = pd.DataFrame({"precision": pc_p, "recall": pc_r, "f1": pc_f, "AP": ap, "support": support}, index=CLASSES)
    summary = {"top1": topk_accuracy(y, scores, 1), "top3": topk_accuracy(y, scores, 3), "top5": topk_accuracy(y, scores, 5),
               "macro_precision": float(p), "macro_recall": float(r), "macro_f1": float(f1), "mAP": float(np.nanmean(ap)), "n": int(len(y))}
    return summary, per_class


def macro_f1(y, scores):
    return f1_score(y, scores.argmax(1), average="macro", zero_division=0)


def plot_confusion(y, pred, title, name):
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
    for ax, m, fmt, sub in [(axes[0], cm, "d", "counts"), (axes[1], cmn, ".2f", "row-normalised (diagonal = recall)")]:
        im = ax.imshow(m, cmap="Blues")
        ax.set_xticks(range(K), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(K), CLASSES); ax.grid(False)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, format(m[i, j], fmt), ha="center", va="center", fontsize=8,
                        color="white" if m[i, j] > m.max() / 2 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{title}: {sub}")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); savefig(fig, f"{name}_confusion"); plt.show()
    off = [(CLASSES[i], CLASSES[j], int(cm[i, j])) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0]
    return pd.DataFrame(sorted(off, key=lambda t: -t[2])[:8], columns=["true", "predicted", "count"])


def show_predictions(frame, y, scores, correct, title, name, n=12, is_prob=True):
    pred = scores.argmax(1)
    idx = np.flatnonzero((pred == y) if correct else (pred != y))
    if len(idx) == 0:
        print("no", "correct" if correct else "wrong", "predictions"); return
    idx = np.random.default_rng(SEED).choice(idx, min(n, len(idx)), replace=False)
    cols = 6
    rows = int(np.ceil(len(idx) / cols))
    row_h = 3.3 if correct else 3.7          # room for the 2- or 3-line titles above each image
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, row_h * rows), squeeze=False)
    tag = "p" if is_prob else "s"
    for ax, i in zip(axes.ravel(), idx):
        top = np.argsort(-scores[i])[:2]
        ax.imshow(Image.open(frame.path[i]))
        txt = f"T: {CLASSES[y[i]]}\nP: {CLASSES[top[0]]} ({tag}={scores[i, top[0]]:.2f})"
        if not correct:
            txt += f"\n2nd: {CLASSES[top[1]]} ({scores[i, top[1]]:.2f})"
        ax.set_title(txt, fontsize=8, color="green" if correct else "red")
    for ax in axes.ravel(): ax.axis("off")
    finish(fig, title, name)


def save_predictions(name, split, y, scores):
    out = splits[split][["relpath", "label"]].copy()
    out["y_true"], out["y_pred"] = y, scores.argmax(1)
    out["pred_label"] = [CLASSES[i] for i in out.y_pred]
    for c in range(K):
        out[f"score_{CLASSES[c]}"] = scores[:, c]
    out.to_csv(OUT / name / f"predictions_{split}.csv", index=False)


def report_model(name, y_val, s_val, y_test, s_test, config, is_prob=True):
    """Final evaluation of a chosen model: val + test metrics, confusion matrix, examples, saved files."""
    (OUT / name).mkdir(exist_ok=True)
    sv, _ = compute_metrics(y_val, s_val)
    st, per_class = compute_metrics(y_test, s_test)
    RESULTS[name] = {"val": sv, "test": st, "config": config}
    display(pd.DataFrame([sv, st], index=["val", "test"]))
    display(per_class)
    display(plot_confusion(y_test, s_test.argmax(1), f"{name} (test)", name))
    show_predictions(splits["test"], y_test, s_test, True, f"{name}: correct test predictions", f"{name}_correct", is_prob=is_prob)
    show_predictions(splits["test"], y_test, s_test, False, f"{name}: wrong test predictions", f"{name}_wrong", is_prob=is_prob)
    save_predictions(name, "val", y_val, s_val)
    save_predictions(name, "test", y_test, s_test)
    per_class.to_csv(OUT / name / "per_class_test.csv")
    with open(OUT / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, "model": name, "seed": SEED, "run_mode": RUN_MODE, **RESULTS[name]}, f, indent=2, default=str)


def save_final(name, kind, model, arch=None, metrics=None):
    """Save a model as results/<dataset>/<name>/model.{pt,joblib} + config.json: the chosen model (after report_model) or any tried config.
    kind: "cnn" | "resnet50" | "clip" (torch, state_dict + architecture) or "sklearn" (whole fitted pipeline)."""
    import joblib
    d = OUT / name
    d.mkdir(parents=True, exist_ok=True)
    arch = json.loads(json.dumps(arch or {}, default=str))   # plain Python types, so torch.load(weights_only=True) can read it
    metrics = metrics or RESULTS.get(name, {})
    if kind == "sklearn":
        file = "model.joblib"
        joblib.dump(model, d / file)
    elif kind == "clip":
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES, "text_emb": model.text_emb.cpu()}, d / file)
    else:
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES,
                    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, d / file)
    config = {"dataset": DATASET, "model": name, "kind": kind, "file": file, "arch": arch, "classes": CLASSES,
              "preprocessing": {"pad_to_square_fill": PAD_FILL, "img_size": IMG_SIZE,
                                "normalize": "clip" if kind == "clip" else "imagenet",
                                "classical_features": CLASSICAL_FEATURES if kind == "sklearn" else None},
              "seed": SEED, "run_mode": RUN_MODE, "torch": torch.__version__,
              "val_macro_f1": metrics.get("val", {}).get("macro_f1"),
              "test_macro_f1": metrics.get("test", {}).get("macro_f1"),
              "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(d / "config.json", "w") as f:
        json.dump(config, f, indent=2, default=str)
    size = (d / file).stat().st_size / 1e6
    print(f"saved {d / file} ({size:.1f} MB) + config.json")


ALL_DIR = OUT / "all_models"   # every config tried: model + config.json + metrics.json + scores.npz
ALL_DIR.mkdir(exist_ok=True)
CONFIGS, CHOSEN = {}, {}       # config name -> its val/test metrics; model type -> the config chosen on val


def config_done(name, arch):
    """Metrics of a config already trained, scored and saved with this arch, else None (then it is trained / scored now)."""
    p = ALL_DIR / name / "metrics.json"
    if RETRAIN or not p.exists() or not (ALL_DIR / name / "scores.npz").exists():
        return None
    res = json.loads(p.read_text())
    if res.get("arch") != json.loads(json.dumps(arch, default=str)):
        return None
    CONFIGS[name] = res
    return res


def record_config(name, family, kind, model, arch, scores, train_f1):
    """Score one tried config on val and test, save it to all_models/<name>/ and remember it.
    scores: {"val": (y, s), "test": (y, s)}. Test is recorded for the report; choosing uses val only."""
    arch = json.loads(json.dumps(arch, default=str))
    res = {"model": name, "family": family, "kind": kind, "arch": arch, "train_f1": float(train_f1),
           **{sp: compute_metrics(*scores[sp])[0] for sp in ("val", "test")}}
    save_final(f"all_models/{name}", kind, model, arch, metrics=res)
    np.savez(ALL_DIR / name / "scores.npz", **{f"{sp}_{k}": v for sp, (y, s) in scores.items() for k, v in (("y", y), ("s", s))})
    with open(ALL_DIR / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, **res}, f, indent=2, default=str)
    CONFIGS[name] = res
    return res


def config_scores(name):
    """(y_val, s_val, y_test, s_test) of a recorded config, read back from all_models/<name>/scores.npz."""
    z = np.load(ALL_DIR / name / "scores.npz")
    return z["val_y"], z["val_s"], z["test_y"], z["test_s"]

In [ ]:
# shared PyTorch training loop (CNN and ResNet-50)
SCALER = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def set_train_mode(model):
    """train() but keep frozen BatchNorm layers in eval mode, so a frozen backbone's running stats don't change."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    set_train_mode(model) if training else model.eval()
    total, ys, ps = 0.0, [], []
    for x, y_cpu in loader:
        # non_blocking copies only on CUDA: on MPS with torch <= 2.12 they can deliver corrupted labels
        x, y = x.to(DEVICE, non_blocking=NON_BLOCKING), y_cpu.to(DEVICE, non_blocking=NON_BLOCKING)
        with torch.set_grad_enabled(training), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = F.cross_entropy(logits, y, label_smoothing=LABEL_SMOOTHING if training else 0.0)
        if training:
            optimizer.zero_grad(set_to_none=True)
            SCALER.scale(loss).backward()
            SCALER.step(optimizer)
            SCALER.update()
        total += loss.item() * len(y)
        ys.append(y_cpu); ps.append(logits.detach().float().softmax(1).cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return total / len(y), y, p


def fit(name, build, cfg, max_epochs, patience, lr, weight_decay):
    """Train with AdamW + cosine LR, keep the epoch with the best val macro-F1, stop after `patience` epochs without gain.
    Checkpoint files carry a hash of the config, so changing a grid value never reloads a stale model."""
    tag = hashlib.md5(str((cfg, max_epochs, patience, lr, weight_decay, LABEL_SMOOTHING, IMG_SIZE)).encode()).hexdigest()[:8]
    ckpt, hist_path = CKPT_DIR / f"{name}_{tag}.pt", CKPT_DIR / f"{name}_{tag}_history.csv"
    seed_everything()
    model = build().to(DEVICE)
    if ckpt.exists() and hist_path.exists() and not RETRAIN:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"{name}: loaded saved model {ckpt.name}")
        return model, pd.read_csv(hist_path), ckpt
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {n_train / 1e6:.2f}M trainable params")
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    train_dl, val_dl = make_loader("train", train_tf, shuffle=True), make_loader("val", eval_tf)
    rows, best, best_epoch, wait = [], -1.0, 0, 0
    for epoch in range(1, max_epochs + 1):
        t0, lr_now = time.time(), opt.param_groups[0]["lr"]
        tr_loss, tr_y, tr_p = run_epoch(model, train_dl, opt)
        va_loss, va_y, va_p = run_epoch(model, val_dl)
        sched.step()
        row = dict(epoch=epoch, lr=lr_now, train_loss=tr_loss, val_loss=va_loss,
                   train_acc=(tr_p.argmax(1) == tr_y).mean(), val_acc=(va_p.argmax(1) == va_y).mean(),
                   train_f1=macro_f1(tr_y, tr_p), val_f1=macro_f1(va_y, va_p), sec=time.time() - t0)
        rows.append(row)
        print(f"  ep {epoch:2d} | loss {tr_loss:.3f}/{va_loss:.3f} | F1 {row['train_f1']:.3f}/{row['val_f1']:.3f} | {row['sec']:.0f}s")
        if row["val_f1"] > best:
            best, best_epoch, wait = row["val_f1"], epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience:
                print(f"  early stop: no val gain for {patience} epochs")
                break
    hist = pd.DataFrame(rows)
    hist["best"] = hist.epoch == best_epoch
    hist.to_csv(hist_path, index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, hist, ckpt


def load_trained(ckpt, build):
    model = build().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model.eval()


@torch.no_grad()
def predict(model, split, transform=None):
    _, y, p = run_epoch(model, make_loader(split, transform or eval_tf))
    return y, p


def plot_learning_curves(hists, title, name):
    """One row per config: loss and macro-F1 per epoch, train vs val, dashed line at the kept epoch."""
    fig, axes = plt.subplots(len(hists), 2, figsize=(12, 3 * len(hists)), squeeze=False)
    for row, (cfg_name, h) in zip(axes, hists.items()):
        best = h.loc[h.best, "epoch"].iloc[0]
        for ax, metric in zip(row, ("loss", "f1")):
            ax.plot(h.epoch, h[f"train_{metric}"], "o-", ms=3, label="train")
            ax.plot(h.epoch, h[f"val_{metric}"], "o-", ms=3, label="val")
            ax.axvline(best, ls="--", c="gray", lw=1)
            ax.set_title(f"{cfg_name}: {'loss' if metric == 'loss' else 'macro-F1'} (kept epoch {best})", fontsize=10)
            ax.set_xlabel("epoch"); ax.legend()
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    finish(fig, title, name)


def plot_grid_summary(table, x, title, name, logx=False):
    """Train vs val macro-F1 at the kept epoch for each config: the 'elbow' plot."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(table[x].astype(str) if not logx else table[x], table.train_f1, "o-", label="train")
    ax.plot(table[x].astype(str) if not logx else table[x], table.val_f1, "o-", label="val")
    if logx: ax.set_xscale("log")
    best = table.val_f1.idxmax()
    ax.scatter([table[x].astype(str)[best] if not logx else table[x][best]], [table.val_f1[best]], s=160,
               facecolors="none", edgecolors="red", lw=2, label="chosen (best val)", zorder=5)
    ax.set_xlabel(x); ax.set_ylabel("macro-F1"); ax.set_title(title); ax.legend()
    plt.tight_layout(); savefig(fig, name); plt.show()

In [ ]:
import cv2, joblib
from joblib import Parallel, delayed
from scipy.stats import binomtest
from skimage.feature import hog, local_binary_pattern
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning


def base_scores(model, split):
    """Scores of a baseline model from its predictions CSV, in this notebook's row order."""
    p = pd.read_csv(BASE_OUT / model / f"predictions_{split}.csv").set_index("relpath").loc[splits[split].relpath]
    assert (p.y_true.values == y_split[split]).all()
    return p[[f"score_{c}" for c in CLASSES]].values


def paired(name_a, s_a, name_b, s_b, y, n_boot=1000):
    """McNemar test + bootstrap 95% CI of the macro-F1 difference (a - b) on the same test images."""
    pa, pb = s_a.argmax(1), s_b.argmax(1)
    a_only, b_only = int(((pa == y) & (pb != y)).sum()), int(((pa != y) & (pb == y)).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    rng = np.random.default_rng(0)
    d = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        d.append(f1_score(y[i], pa[i], average="macro") - f1_score(y[i], pb[i], average="macro"))
    delta = f1_score(y, pa, average="macro") - f1_score(y, pb, average="macro")
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"improved": name_a, "baseline": name_b, "improved_only_right": a_only, "baseline_only_right": b_only,
            "mcnemar_p": p, "delta_f1": delta, "ci_lo": lo, "ci_hi": hi, "significant": bool(p < 0.05 and (lo > 0 or hi < 0))}


def choose(prefix_or_names):
    """Config with the best val macro-F1 among the recorded ones."""
    names = [n for n in CONFIGS if n.startswith(prefix_or_names)] if isinstance(prefix_or_names, str) else prefix_or_names
    return max(names, key=lambda n: CONFIGS[n]["val"]["macro_f1"])


PAIRS = []   # (improved model, baseline model) pairs tested at the end

## A. Classic SVM

**Why the baseline is stuck near 0.50.** Train F1 reaches 1.0 while val stays at 0.50, and C / gamma barely change val, so the features are the limit.
The baseline vector is 512 colour + 1,764 HOG + 28 LBP numbers. After `StandardScaler` every number weighs the same, so HOG decides 77% of the RBF distance, LBP almost nothing.

- **A0 ablation**: the baseline's own cached features (no re-extraction), each group alone, each pair, all three.
- **A1 balanced groups**: after scaling, each group is divided by √(its size), so each group contributes the same total variance.
- **A2 `handcrafted_v2`**: hue×saturation and LAB a\*b\* histograms (brightness left out: these classes are dark, so V bins are crowded, and no enhancement is added so 01 vs 02 stays clean),
  LBP at 3 radii on the whole image and on a 2×2 grid, a Gabor filter bank, the same HOG; square root on the histograms (Hellinger), then balanced groups.

gamma is set as `gamma_mult / total variance after scaling`, which equals the baseline's `gamma_mult / n_features` when no weights are used.

In [ ]:
class GroupScale(BaseEstimator, TransformerMixin):
    """Multiply each column by its group weight (after StandardScaler)."""
    def __init__(self, weights=None):
        self.weights = weights

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X if self.weights is None else X * np.asarray(self.weights, dtype=np.float32)


def make_svm2(C, gamma_mult, weights=None, n_cols=None):
    total_var = float(np.sum(np.square(weights))) if weights is not None else n_cols
    return make_pipeline(StandardScaler(), GroupScale(weights),
                         SVC(C=C, gamma=gamma_mult / total_var, kernel="rbf", decision_function_shape="ovr"))


def group_weights(groups, balanced):
    """groups: list of (name, slice) in column order -> per-column weights (1/sqrt(size) per group) or None."""
    if not balanced:
        return None
    return np.concatenate([np.full(s.stop - s.start, 1 / np.sqrt(s.stop - s.start)) for _, s in groups])


def svm_sweep(feats, feat_name, groups, balanced, Cs, gms, tag):
    """Fit an SVM per (C, gamma_mult) on the chosen column groups; every config is recorded like the team's sweep."""
    cols = np.concatenate([np.arange(s.start, s.stop) for _, s in groups])
    Xs = {sp: feats[sp][:, cols] for sp in ("train", "val", "test")}
    sizes, start, local = [], 0, []
    for g, s in groups:                                   # the same groups, re-indexed inside the selected columns
        n = s.stop - s.start; local.append((g, slice(start, start + n))); start += n
    w = group_weights(local, balanced)
    rows = []
    for C in Cs:
        for gm in gms:
            name = f"svm2_{tag}_C{C:g}_g{gm:g}"
            arch = {"features": feat_name, "groups": [g for g, _ in groups], "balanced": balanced, "C": C, "gamma_mult": gm}
            res = config_done(name, arch)
            if res is None:
                t0 = time.time()
                m = make_svm2(C, gm, w, len(cols)).fit(Xs["train"], y_split["train"])
                res = record_config(name, "svm", "sklearn", m, arch,
                                    {sp: (y_split[sp], m.decision_function(Xs[sp])) for sp in ("val", "test")},
                                    train_f1=macro_f1(y_split["train"], m.decision_function(Xs["train"])))
                print(f"  {name}: train {res['train_f1']:.3f} | val {res['val']['macro_f1']:.3f} | {time.time() - t0:.0f}s")
            rows.append({"tag": tag, "C": C, "gamma_mult": gm, "name": name, "train_f1": res["train_f1"], "val_f1": res["val"]["macro_f1"]})
    return pd.DataFrame(rows)


def load_base_features(split):
    z = np.load(BASE_OUT / f"features_handcrafted_{split}.npz", allow_pickle=True)
    assert list(z["relpath"]) == splits[split].relpath.tolist(), "baseline feature cache is for other images"
    return z["X"]


XB = {sp: load_base_features(sp) for sp in ("train", "val", "test")}
assert XB["train"].shape[1] == 512 + 1764 + 28
BASE_GROUPS = {"color": slice(0, 512), "hog": slice(512, 2276), "lbp": slice(2276, 2304)}
print({sp: v.shape for sp, v in XB.items()})

In [ ]:
# A0: which feature group carries the signal? (baseline features, unweighted, gamma x1)
subsets = [["color"], ["hog"], ["lbp"], ["color", "hog"], ["color", "lbp"], ["hog", "lbp"], ["color", "hog", "lbp"]]
abl = pd.concat([svm_sweep(XB, "handcrafted", [(g, BASE_GROUPS[g]) for g in sub], False, ABLATION_C, [1], "+".join(sub))
                 for sub in subsets], ignore_index=True)
abl_best = abl.loc[abl.groupby("tag").val_f1.idxmax()].set_index("tag").loc[["+".join(s) for s in subsets]]
display(abl_best)

check = abl[(abl.tag == "color+hog+lbp") & (abl.C == 10)]
if RUN_MODE == "full" and len(check):
    base_val = json.loads((BASE_OUT / "all_models" / "svm_C10_g1" / "metrics.json").read_text())["val"]["macro_f1"]
    print(f"sanity: all groups, C=10, gamma x1 -> val {check.val_f1.iloc[0]:.4f} (baseline svm_C10_g1: {base_val:.4f})")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(abl_best.index, abl_best.val_f1, color=["C0", "C1", "C2", "C3", "C4", "C5", "C7"])
for i, (v, t) in enumerate(zip(abl_best.val_f1, abl_best.train_f1)):
    ax.text(i, v, f"{v:.3f}\n(train {t:.2f})", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("val macro-F1 (best C)"); ax.set_ylim(0, max(0.8, abl_best.val_f1.max() + 0.15))
ax.set_title("A0: SVM on each baseline feature group"); plt.xticks(rotation=20)
plt.tight_layout(); savefig(fig, "svm_ablation"); plt.show()

In [ ]:
# A1: the same baseline features, each group weighted 1/sqrt(size)
a1 = svm_sweep(XB, "handcrafted", list(BASE_GROUPS.items()), True, SVM2_C, SVM2_GAMMA_MULT, "balanced")
display(a1.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))

In [ ]:
# A2: handcrafted_v2 features
GABOR = [cv2.getGaborKernel((21, 21), sigma=0.56 * lam, theta=th, lambd=lam, gamma=0.5, psi=0)
         for lam in (4, 8, 16) for th in np.arange(4) * np.pi / 4]


def handcrafted_v2(path):
    rgb = np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((128, 128), Image.BILINEAR))   # same input as the baseline
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hs = cv2.calcHist([hsv], [0, 1], None, [18, 8], [0, 180, 0, 256]).ravel()                # hue x saturation, brightness left out
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    ab = cv2.calcHist([lab], [1, 2], None, [16, 16], [0, 256, 0, 256]).ravel()               # a*b* colour, L left out
    color = np.concatenate([hs / (hs.sum() + 1e-8), ab / (ab.sum() + 1e-8)])
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")   # as the baseline
    texture = []
    for p, r in ((8, 1), (16, 2), (24, 3)):
        codes = local_binary_pattern(gray, p, r, method="uniform")
        for part in (codes, codes[:64, :64], codes[:64, 64:], codes[64:, :64], codes[64:, 64:]):   # whole image + 2x2 grid
            h, _ = np.histogram(part, bins=p + 2, range=(0, p + 2))
            texture.append(h / h.sum())
    g = gray.astype(np.float32) / 255
    gabor = []
    for k in GABOR:
        resp = np.abs(cv2.filter2D(g, cv2.CV_32F, k))
        gabor += [resp.mean(), resp.std()]
    color, texture = np.sqrt(color), np.sqrt(np.concatenate(texture))                          # Hellinger for histograms
    return np.concatenate([color, shape, texture, np.array(gabor)]).astype(np.float32)


V2_GROUPS = {"color": slice(0, 144 + 256), "hog": slice(400, 400 + 1764), "lbp": slice(2164, 2164 + 270), "gabor": slice(2434, 2434 + 24)}


def load_v2(split):
    cache = OUT / f"features_v2_{split}.npz"
    if cache.exists() and not RETRAIN:
        z = np.load(cache, allow_pickle=True)
        if list(z["relpath"]) == splits[split].relpath.tolist():
            return z["X"]
    t0 = time.time()
    Xv = np.stack(Parallel(n_jobs=-1, batch_size=32)(delayed(handcrafted_v2)(p) for p in splits[split].path))
    np.savez(cache, X=Xv, relpath=splits[split].relpath.values)
    print(f"{split}: v2 features {Xv.shape} in {time.time() - t0:.0f}s")
    return Xv


XV2 = {sp: load_v2(sp) for sp in ("train", "val", "test")}
assert XV2["train"].shape[1] == V2_GROUPS["gabor"].stop
a2 = svm_sweep(XV2, "handcrafted_v2", list(V2_GROUPS.items()), True, SVM2_C, SVM2_GAMMA_MULT, "v2")
display(a2.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))
a2_nohog = svm_sweep(XV2, "handcrafted_v2", [(g, V2_GROUPS[g]) for g in ("color", "lbp", "gabor")], True, SVM2_C, SVM2_GAMMA_MULT, "v2nohog")
display(a2_nohog.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))

In [ ]:
# chosen classic model: best val over A0-A2
best = choose("svm2_")
CHOSEN["svm_improved"] = best
print("chosen:", best, CONFIGS[best]["arch"])
report_model("svm_improved", *config_scores(best), CONFIGS[best]["arch"], is_prob=False)
PAIRS.append(("svm_improved", "svm"))

## B. CLIP

- **B1** CLIP ViT-L/14 next to the baseline ViT-B/32, same 4 template sets, plus a 5th set **"descriptions"**: one short visual description per class
  (written from general knowledge before looking at any errors), averaged with the "food photo" template.
- **B2** flip test-time augmentation: average the embeddings of the image and its mirror image (for the best zero-shot setting).
- **B3** linear probe: logistic regression on the frozen image embeddings. It uses the 5,780 training images, so it is a new **"few-shot"** row, not zero-shot.

In [ ]:
from transformers import CLIPModel, AutoTokenizer

DESCRIPTIONS = {
    "baby_back_ribs": "a photo of baby back ribs, a rack of pork ribs glazed with barbecue sauce.",
    "bibimbap": "a photo of bibimbap, a Korean rice bowl topped with vegetables, a fried egg and red chili paste.",
    "chicken_wings": "a photo of chicken wings, fried or glazed wing pieces often served with a dipping sauce.",
    "edamame": "a photo of edamame, green soybean pods sprinkled with salt.",
    "guacamole": "a photo of guacamole, a green mashed avocado dip served with tortilla chips.",
    "miso_soup": "a photo of miso soup, a cloudy Japanese broth with tofu cubes and seaweed in a bowl.",
    "mussels": "a photo of mussels, black-shelled shellfish steamed in broth.",
    "nachos": "a photo of nachos, tortilla chips covered with melted cheese and toppings.",
    "paella": "a photo of paella, a Spanish saffron rice dish with seafood in a wide shallow pan.",
    "prime_rib": "a photo of prime rib, a thick slice of roast beef with a pink center.",
}
PROMPTS = {k: {c: [t.format(c.replace("_", " ")) for t in v] for c in CLASSES} for k, v in TEMPLATE_SETS.items()}
PROMPTS["descriptions"] = {c: [TEMPLATE_SETS["food photo"][0].format(c.replace("_", " ")), DESCRIPTIONS[c]] for c in CLASSES}

hflip = T.Compose([T.Lambda(lambda im: im.transpose(Image.FLIP_LEFT_RIGHT)), clip_tf])


class TextHead(nn.Module):
    """Holds the class text embeddings, so save_final(kind="clip") can store them."""
    def __init__(self, text_emb):
        super().__init__()
        self.register_buffer("text_emb", text_emb)


@torch.no_grad()
def text_emb(model, tok, prompts):
    out = []
    for c in CLASSES:
        t = tok(prompts[c], padding=True, return_tensors="pt").to(DEVICE)
        e = model.text_projection(model.text_model(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).pooler_output)
        out.append(F.normalize(F.normalize(e, dim=-1).mean(0), dim=0))
    return torch.stack(out)


@torch.no_grad()
def image_emb(model, key, split, flip=False):
    cache = OUT / f"clip{key}_img_{split}{'_flip' if flip else ''}.pt"
    if cache.exists() and not RETRAIN:
        e = torch.load(cache)
        if len(e) == len(splits[split]):
            return e
    t0, feats = time.time(), []
    for x, _ in make_loader(split, hflip if flip else clip_tf):
        f = model.visual_projection(model.vision_model(pixel_values=x.to(DEVICE)).pooler_output)
        feats.append(F.normalize(f, dim=-1).cpu())
    e = torch.cat(feats)
    torch.save(e, cache)
    print(f"  clip{key} {split}{' flip' if flip else ''}: {tuple(e.shape)} in {time.time() - t0:.0f}s")
    return e


CLIP_EMB, CLIP_TXT, CLIP_SCALE = {}, {}, {}
for key, cid in CLIP_IDS.items():
    m = CLIPModel.from_pretrained(cid).to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(cid)
    CLIP_SCALE[key] = float(m.logit_scale.exp())
    CLIP_TXT[key] = {k: text_emb(m, tok, p).cpu() for k, p in PROMPTS.items()}
    CLIP_EMB[key] = {sp: image_emb(m, key, sp) for sp in ("train", "val", "test")}
    CLIP_EMB[key].update({f"{sp}_flip": image_emb(m, key, sp, flip=True) for sp in ("val", "test")})
    del m
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

In [ ]:
# B1 + B2: zero-shot (nothing is trained)
def zs_scores(key, tname, split, tta=False):
    e = CLIP_EMB[key][split]
    if tta:
        e = F.normalize(e + CLIP_EMB[key][f"{split}_flip"], dim=-1)
    return (CLIP_SCALE[key] * e @ CLIP_TXT[key][tname].T).softmax(1).numpy()


def record_zs(key, tname, tta=False):
    name = f"clip{key}_{tname.replace(' ', '_')}{'_tta' if tta else ''}"
    arch = {"clip_id": CLIP_IDS[key], "prompts": tname, "tta": tta}
    res = config_done(name, arch)
    if res is None:
        res = record_config(name, "clip_zeroshot", "clip", TextHead(CLIP_TXT[key][tname]), arch,
                            {sp: (y_split[sp], zs_scores(key, tname, sp, tta)) for sp in ("val", "test")}, train_f1=float("nan"))
    return {"backbone": key, "prompts": tname, "tta": tta, "name": name, "val_f1": res["val"]["macro_f1"]}


zs = pd.DataFrame([record_zs(k, t) for k in CLIP_IDS for t in PROMPTS])
display(zs.pivot(index="prompts", columns="backbone", values="val_f1").style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))
top = zs.loc[zs.val_f1.idxmax()]
zs = pd.concat([zs, pd.DataFrame([record_zs(top.backbone, top.prompts, tta=True)])], ignore_index=True)
display(zs.sort_values("val_f1", ascending=False).head(6))

fig, ax = plt.subplots(figsize=(9, 3.8))
piv = zs[~zs.tta].pivot(index="prompts", columns="backbone", values="val_f1")
piv.plot.bar(ax=ax); ax.set_ylim(0.85, 1); ax.set_ylabel("val macro-F1"); ax.set_title("CLIP zero-shot: backbone x prompts (val)")
plt.xticks(rotation=15); plt.tight_layout(); savefig(fig, "clip_backbone_prompts"); plt.show()

best = choose([n for n in CONFIGS if n.startswith("clip") and "_probe_" not in n])
CHOSEN["clip_zeroshot_improved"] = best
print("chosen zero-shot:", best, CONFIGS[best]["arch"])
report_model("clip_zeroshot_improved", *config_scores(best), CONFIGS[best]["arch"])
PAIRS.append(("clip_zeroshot_improved", "clip_zeroshot"))

In [ ]:
# B3: linear probe on frozen CLIP image embeddings (uses the training images: a new "few-shot" row)
rows = []
for key in CLIP_IDS:
    E = {sp: CLIP_EMB[key][sp].numpy() for sp in ("train", "val", "test")}
    for C in PROBE_C:
        name, arch = f"clip{key}_probe_C{C:g}", {"clip_id": CLIP_IDS[key], "probe": "logreg", "C": C}
        res = config_done(name, arch)
        if res is None:
            m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(E["train"], y_split["train"])
            res = record_config(name, "clip_probe", "sklearn", m, arch,
                                {sp: (y_split[sp], m.predict_proba(E[sp])) for sp in ("val", "test")},
                                train_f1=macro_f1(y_split["train"], m.predict_proba(E["train"])))
        rows.append({"backbone": key, "C": C, "name": name, "train_f1": res["train_f1"], "val_f1": res["val"]["macro_f1"]})
probe = pd.DataFrame(rows)
display(probe.pivot(index="C", columns="backbone", values="val_f1").style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))

best = choose([n for n in CONFIGS if "_probe_" in n])
CHOSEN["clip_linear_probe"] = best
print("chosen probe:", best)
report_model("clip_linear_probe", *config_scores(best), CONFIGS[best]["arch"])
PAIRS.append(("clip_linear_probe", "clip_zeroshot"))

## C. Ensemble: CLIP + ResNet-50

The two make different mistakes (baseline: 87 and 94 wrong test images, only 39 in common).
Scores are combined as `w·log p_CLIP + (1−w)·log p_ResNet` (CLIP's softmax is very peaked, so averaging plain probabilities would let CLIP decide alone); `w` is chosen on val.

In [ ]:
def ensemble(name, sa, sb, label_a, label_b):
    """sa, sb: {"val": scores, "test": scores}. Picks w on val, records every w, returns the chosen name."""
    rows = []
    for w in np.round(np.arange(0, 1.01, 0.1), 1):
        cname, arch = f"{name}_w{w:g}", {"a": label_a, "b": label_b, "w": float(w)}
        res = config_done(cname, arch)
        if res is None:
            mix = {sp: torch.from_numpy(w * np.log(sa[sp] + 1e-9) + (1 - w) * np.log(sb[sp] + 1e-9)).softmax(1).numpy() for sp in ("val", "test")}
            res = record_config(cname, "ensemble", "sklearn", {"w": float(w), "a": label_a, "b": label_b}, arch,
                                {sp: (y_split[sp], mix[sp]) for sp in ("val", "test")}, train_f1=float("nan"))
        rows.append({"w_clip": w, "name": cname, "val_f1": res["val"]["macro_f1"], "test_f1": res["test"]["macro_f1"]})
    t = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    ax.plot(t.w_clip, t.val_f1, "o-", label="val")
    b = t.val_f1.idxmax()
    ax.scatter([t.w_clip[b]], [t.val_f1[b]], s=160, facecolors="none", edgecolors="red", lw=2, label="chosen", zorder=5)
    ax.set_xlabel(f"weight on {label_a}"); ax.set_ylabel("val macro-F1"); ax.set_title(f"{name}: {label_a} + {label_b}"); ax.legend()
    plt.tight_layout(); savefig(fig, f"{name}_weight"); plt.show()
    return t.name[b]


rn_base = {sp: base_scores("resnet50", sp) for sp in ("val", "test")}
clip_base = {sp: base_scores("clip_zeroshot", sp) for sp in ("val", "test")}
best_zs = config_scores(CHOSEN["clip_zeroshot_improved"])
clip_new = {"val": best_zs[1], "test": best_zs[3]}

e1 = ensemble("ens_clipB32_rn50", clip_base, rn_base, "CLIP B/32 (baseline)", "ResNet-50 (baseline)")
e2 = ensemble("ens_clipbest_rn50", clip_new, rn_base, CHOSEN["clip_zeroshot_improved"], "ResNet-50 (baseline)")
best = choose([e1, e2])
CHOSEN["ensemble"] = best
print("chosen ensemble:", best, CONFIGS[best]["arch"])
report_model("ensemble", *config_scores(best), CONFIGS[best]["arch"])
PAIRS += [("ensemble", "resnet50"), ("ensemble", "clip_zeroshot")]

## Training setup for D and E

- **Stronger augmentation** (both D and E): the baseline's random crop + flip, plus `ColorJitter(0.3, 0.3, 0.2)` and `RandAugment(2, 9)`.
  It is set by replacing `train_tf`, which the team's `fit()` reads when it builds the loader.
- **`fit_lw`** (E only): a copy of the team's `fit()` with two AdamW parameter groups (pretrained backbone at lr 1e-4, new head at 1e-3)
  and a 1-epoch warmup at lr/10 before the cosine decay. Everything else (best-val checkpoint, early stopping, history CSV) is unchanged.

In [ ]:
def make_tfs(size, strong):
    aug = [T.ColorJitter(0.3, 0.3, 0.2), T.RandAugment(2, 9)] if strong else []
    tr = T.Compose([T.Lambda(pad_to_square), T.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(),
                    *aug, T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    ev = T.Compose([T.Lambda(pad_to_square), T.Resize((size, size)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return tr, ev


def fit_lw(name, build, cfg, max_epochs, patience, lr_backbone, lr_head, weight_decay, head_prefix="fc."):
    """The team's fit() with layer-wise learning rates and a 1-epoch warmup."""
    tag = hashlib.md5(str((cfg, max_epochs, patience, lr_backbone, lr_head, weight_decay, LABEL_SMOOTHING, IMG_SIZE)).encode()).hexdigest()[:8]
    ckpt, hist_path = CKPT_DIR / f"{name}_{tag}.pt", CKPT_DIR / f"{name}_{tag}_history.csv"
    seed_everything()
    model = build().to(DEVICE)
    if ckpt.exists() and hist_path.exists() and not RETRAIN:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"{name}: loaded saved model {ckpt.name}")
        return model, pd.read_csv(hist_path), ckpt
    head = [p for n, p in model.named_parameters() if n.startswith(head_prefix)]
    body = [p for n, p in model.named_parameters() if not n.startswith(head_prefix)]
    opt = torch.optim.AdamW([{"params": body, "lr": lr_backbone}, {"params": head, "lr": lr_head}], weight_decay=weight_decay)
    warm = lambda e: 0.1 if e == 0 else 0.5 * (1 + np.cos(np.pi * (e - 1) / max(1, max_epochs - 1)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, [warm, warm])
    print(f"{name}: {sum(p.numel() for p in body + head) / 1e6:.2f}M trainable params")
    train_dl, val_dl = make_loader("train", train_tf, shuffle=True), make_loader("val", eval_tf)
    rows, best, best_epoch, wait = [], -1.0, 0, 0
    for epoch in range(1, max_epochs + 1):
        t0, lr_now = time.time(), opt.param_groups[0]["lr"]
        tr_loss, tr_y, tr_p = run_epoch(model, train_dl, opt)
        va_loss, va_y, va_p = run_epoch(model, val_dl)
        sched.step()
        row = dict(epoch=epoch, lr=lr_now, train_loss=tr_loss, val_loss=va_loss,
                   train_acc=(tr_p.argmax(1) == tr_y).mean(), val_acc=(va_p.argmax(1) == va_y).mean(),
                   train_f1=macro_f1(tr_y, tr_p), val_f1=macro_f1(va_y, va_p), sec=time.time() - t0)
        rows.append(row)
        print(f"  ep {epoch:2d} | loss {tr_loss:.3f}/{va_loss:.3f} | F1 {row['train_f1']:.3f}/{row['val_f1']:.3f} | {row['sec']:.0f}s")
        if row["val_f1"] > best:
            best, best_epoch, wait = row["val_f1"], epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience:
                print(f"  early stop: no val gain for {patience} epochs")
                break
    hist = pd.DataFrame(rows)
    hist["best"] = hist.epoch == best_epoch
    hist.to_csv(hist_path, index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, hist, ckpt


def make_resnet50_all():
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.fc.in_features, K))   # same head as the baseline
    return m


class SimpleCNN(nn.Module):   # the team's CNN (cell 25), unchanged
    def __init__(self, n_blocks=4, base=32, dropout=0.3, n_classes=K):
        super().__init__()
        layers, c_in = [], 3
        for i in range(n_blocks):
            c = base * 2 ** i
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


class ResBlock(nn.Module):
    """The same two conv-BN-ReLU layers as SimpleCNN, plus a 1x1 projection skip, then max-pool."""
    def __init__(self, c_in, c):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                                  nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c))
        self.skip = nn.Sequential(nn.Conv2d(c_in, c, 1, bias=False), nn.BatchNorm2d(c))
        self.act, self.pool = nn.ReLU(inplace=True), nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(self.act(self.body(x) + self.skip(x)))


class ResCNN(nn.Module):
    def __init__(self, n_blocks=5, base=32, dropout=0.3, n_classes=K):
        super().__init__()
        self.blocks = nn.Sequential(*[ResBlock(3 if i == 0 else base * 2 ** (i - 1), base * 2 ** i) for i in range(n_blocks)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(base * 2 ** (n_blocks - 1), n_classes))

    def forward(self, x):
        return self.head(self.blocks(x))


TRAINED = {}   # config name -> (model, history, eval transform)


def train_and_record(name, family, kind, build, cfg, fitter, **kw):
    model, hist, ckpt = fitter(name, build, cfg, **kw)
    kept = hist[hist.best].iloc[0]
    if config_done(name, cfg) is None or ckpt.stat().st_mtime > (ALL_DIR / name / "metrics.json").stat().st_mtime:
        record_config(name, family, kind, model.eval(), cfg, {sp: predict(model, sp) for sp in ("val", "test")}, kept.train_f1)
    TRAINED[name] = (model.eval(), hist, eval_tf)
    print(f"{name}: kept epoch {int(kept.epoch)} of {len(hist)} | val {kept.val_f1:.3f} | test {CONFIGS[name]['test']['macro_f1']:.3f}")

## E1. ResNet-50: layer-wise LR + warmup + stronger augmentation (224 px)

In [ ]:
IMG_SIZE = 224
train_tf, eval_tf = make_tfs(IMG_SIZE, strong=True)
train_and_record("resnet50_lw_aug", "resnet50", "resnet50", make_resnet50_all,
                 {"unfreeze": "all", "lr_backbone": 1e-4, "lr_head": 1e-3, "warmup": 1, "aug": "cj+ra", "img": 224},
                 fit_lw, lr_backbone=1e-4, lr_head=1e-3, **RN2_TRAIN)

## D1. CNN: 5 blocks, 100 epochs, stronger augmentation

In [ ]:
IMG_SIZE = 224
train_tf, eval_tf = make_tfs(IMG_SIZE, strong=True)
train_and_record("cnn_d5_long_aug", "cnn", "cnn", lambda: SimpleCNN(n_blocks=5, base=32, dropout=0.3),
                 {"n_blocks": 5, "base": 32, "dropout": 0.3, "aug": "cj+ra"}, fit, **CNN2_TRAIN)

## E2. ResNet-50 at 288 px (same recipe as E1)

In [ ]:
IMG_SIZE = 288
train_tf, eval_tf = make_tfs(IMG_SIZE, strong=True)
train_and_record("resnet50_lw_aug_288", "resnet50", "resnet50", make_resnet50_all,
                 {"unfreeze": "all", "lr_backbone": 1e-4, "lr_head": 1e-3, "warmup": 1, "aug": "cj+ra", "img": 288},
                 fit_lw, lr_backbone=1e-4, lr_head=1e-3, **RN2_TRAIN)
IMG_SIZE = 224
train_tf, eval_tf = make_tfs(IMG_SIZE, strong=True)

## D2. CNN with residual blocks (same training as D1)

In [ ]:
train_and_record("cnn_res5_long_aug", "cnn", "cnn", lambda: ResCNN(n_blocks=5, base=32, dropout=0.3),
                 {"arch": "rescnn", "n_blocks": 5, "base": 32, "dropout": 0.3, "aug": "cj+ra"}, fit, **CNN2_TRAIN)

In [ ]:
# chosen improved CNN and ResNet-50 (best val), learning curves next to the baseline's best run
for fam, names, base in [("cnn", ["cnn_d5_long_aug", "cnn_res5_long_aug"], "cnn_d5"),
                         ("resnet50", ["resnet50_lw_aug", "resnet50_lw_aug_288"], "resnet50_all")]:
    best = choose(names)
    CHOSEN[f"{fam}_improved"] = best
    print(f"{fam}: chosen {best}")
    report_model(f"{fam}_improved", *config_scores(best), CONFIGS[best]["arch"])
    PAIRS.append((f"{fam}_improved", fam))
    base_hist = pd.read_csv(next((BASE_OUT / "checkpoints").glob(f"{base}_*_history.csv")))
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(base_hist.epoch, base_hist.val_f1, "k--", label=f"baseline {base} (val)")
    for n in names:
        h = TRAINED[n][1]
        ax.plot(h.epoch, h.val_f1, "-", label=f"{n} (val)")
        ax.plot(h.epoch, h.train_f1, ":", alpha=0.6, label=f"{n} (train)")
    ax.set_xlabel("epoch"); ax.set_ylabel("macro-F1"); ax.set_title(f"{fam}: improved runs vs baseline"); ax.legend(fontsize=8)
    plt.tight_layout(); savefig(fig, f"{fam}_improved_curves"); plt.show()

In [ ]:
# ensemble again with the improved ResNet-50 (w chosen on val)
rn_new = config_scores(CHOSEN["resnet50_improved"])
e3 = ensemble("ens_clipbest_rn50new", clip_new, {"val": rn_new[1], "test": rn_new[3]},
              CHOSEN["clip_zeroshot_improved"], CHOSEN["resnet50_improved"])
best = choose([CHOSEN["ensemble"], e3])
if best != CHOSEN["ensemble"]:
    CHOSEN["ensemble"] = best
    print("new chosen ensemble:", best, CONFIGS[best]["arch"])
    report_model("ensemble", *config_scores(best), CONFIGS[best]["arch"])

## F. Summary: baseline vs improved (dataset 01, test set)

Each improved row is the config chosen on val among the ones tried above, and is compared with the baseline on the same 1,920 test images
(McNemar test + bootstrap 95% CI of Δ macro-F1). "significant" = p < 0.05 and the CI excludes 0.

In [ ]:
base_sum = pd.read_csv(BASE_OUT / "summary_test.csv", index_col=0)
cols = ["top1", "macro_precision", "macro_recall", "macro_f1", "mAP"]
rows = []
for m in ["clip_zeroshot", "resnet50", "cnn", "svm", "logreg"]:
    rows.append({"model": m, "stage": "baseline", "config": "team notebook", "val_macro_f1": base_sum.loc[m, "val_macro_f1"], **base_sum.loc[m, cols].to_dict()})
for m in ["clip_zeroshot_improved", "clip_linear_probe", "ensemble", "resnet50_improved", "cnn_improved", "svm_improved"]:
    rows.append({"model": m, "stage": "improved", "config": CHOSEN[m], "val_macro_f1": RESULTS[m]["val"]["macro_f1"],
                 **{c: RESULTS[m]["test"][c] for c in cols}})
summary2 = pd.DataFrame(rows).set_index("model")
summary2.to_csv(OUT / "improved_summary.csv")
display(summary2.style.format({c: "{:.4f}" for c in cols + ["val_macro_f1"]}))

y_test = y_split["test"]
tests = pd.DataFrame([paired(a, config_scores(CHOSEN[a])[3], b, base_scores(b, "test"), y_test) for a, b in PAIRS])
tests.to_csv(OUT / "paired_vs_baseline.csv", index=False)
display(tests.style.format({"mcnemar_p": "{:.2g}", "delta_f1": "{:+.4f}", "ci_lo": "{:+.4f}", "ci_hi": "{:+.4f}"}))

pd.DataFrame([{"config": n, "family": r["family"], "arch": json.dumps(r["arch"]), "train_f1": r["train_f1"],
               "val_macro_f1": r["val"]["macro_f1"], "test_macro_f1": r["test"]["macro_f1"], "test_top1": r["test"]["top1"]}
              for n, r in CONFIGS.items()]).to_csv(OUT / "all_configs_improved.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
order = [("svm", "svm_improved"), ("cnn", "cnn_improved"), ("resnet50", "resnet50_improved"), ("clip_zeroshot", "clip_zeroshot_improved")]
xs = np.arange(len(order))
ax.bar(xs - 0.2, [summary2.loc[b, "macro_f1"] for b, _ in order], 0.4, label="baseline")
ax.bar(xs + 0.2, [summary2.loc[i, "macro_f1"] for _, i in order], 0.4, label="improved")
for x, (b, i) in zip(xs, order):
    ax.text(x - 0.2, summary2.loc[b, "macro_f1"], f"{summary2.loc[b, 'macro_f1']:.3f}", ha="center", va="bottom", fontsize=8)
    ax.text(x + 0.2, summary2.loc[i, "macro_f1"], f"{summary2.loc[i, 'macro_f1']:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs, [b for b, _ in order]); ax.set_ylabel("test macro-F1"); ax.set_ylim(0, 1.05); ax.legend()
ax.set_title(f"{DATASET}: baseline vs improved (chosen on val)")
plt.tight_layout(); savefig(fig, "baseline_vs_improved"); plt.show()

## G. Grad-CAM of the improved CNN and ResNet-50 on the baseline's fixed test images

In [ ]:
class GradCAM:   # the team's class (cell 34), unchanged
    def __init__(self, model, layer, tokens=False):
        self.model, self.tokens = model, tokens
        self.handle = layer.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.acts = out
        out.register_hook(lambda g: setattr(self, "grads", g))

    def __call__(self, x):
        self.model.eval()
        x = x.to(DEVICE).requires_grad_(True)
        logits = self.model(x)
        cls = logits.argmax(1)
        self.model.zero_grad(set_to_none=True)
        logits.gather(1, cls[:, None]).sum().backward()
        A, G = self.acts.detach(), self.grads.detach()
        cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = (cam - cam.amin((1, 2), keepdim=True)) / (cam.amax((1, 2), keepdim=True) - cam.amin((1, 2), keepdim=True) + 1e-8)
        return cam.cpu().numpy(), logits.detach().softmax(1).cpu().numpy()

    def remove(self):
        self.handle.remove()


def cam_layer(model):
    if isinstance(model, ResCNN):
        return model.blocks[-1].act
    if isinstance(model, SimpleCNN):
        return model.features[-2]
    return model.layer4


cams_for = {f"{fam}: {CHOSEN[f'{fam}_improved']}": TRAINED[CHOSEN[f"{fam}_improved"]] for fam in ("cnn", "resnet50")}
fixed = pd.read_csv(BASE_OUT / "gradcam_fixed_images.csv").merge(splits["test"], on="relpath")
fig, axes = plt.subplots(len(fixed), 1 + len(cams_for), figsize=(3.2 * (1 + len(cams_for)), 3.1 * len(fixed)), squeeze=False)
cams = {k: GradCAM(m, cam_layer(m)) for k, (m, _, _) in cams_for.items()}
try:
    for row, (_, r) in zip(axes, fixed.iterrows()):
        img = Image.open(r.path).convert("RGB")
        row[0].imshow(np.asarray(pad_to_square(img).resize((224, 224)))); row[0].set_title(f"true: {r.label}", fontsize=9)
        for ax, (k, cam) in zip(row[1:], cams.items()):
            tf = cams_for[k][2]
            heat, prob = cam(tf(img)[None])
            side = heat.shape[-1]
            p = prob[0].argmax()
            ax.imshow(np.asarray(pad_to_square(img).resize((side, side)))); ax.imshow(heat[0], cmap="jet", alpha=0.45)
            ax.set_title(f"{k.split(':')[0]}: {CLASSES[p]} ({prob[0, p]:.2f})", fontsize=9, color="green" if p == r.y else "red")
        for ax in row: ax.axis("off")
finally:
    for c in cams.values(): c.remove()
finish(fig, f"{DATASET}: Grad-CAM of the improved models (fixed test images)", "gradcam_fixed_improved")